# TP 3 — Matrice de cooccurrence et PPMI

## Bibliothèques utilisées

Ce TP utilise NumPy et plusieurs modules de la bibliothèque standard.

- `collections.Counter` est un dictionnaire spécialisé dans le comptage. `Counter(tokens)` calcule les fréquences et `most_common(n)` renvoie les `n` éléments les plus fréquents.
- `re` applique l'expression régulière définie au TP précédent.
- NumPy représente la matrice et applique les calculs à toutes ses cellules : sommes par axe, produit extérieur, logarithme et opérations d'algèbre linéaire.

`Counter` et `re` font partie de Python et n'ont pas besoin d'être installés.

Documentation : [`Counter`](https://docs.python.org/fr/3/library/collections.html#collections.Counter), [guide NumPy pour débuter](https://numpy.org/doc/stable/user/absolute_beginners.html).

In [ ]:
import re
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np

print(np.__version__)

## 1. Reprendre le corpus tokenisé

Le fichier créé au TP précédent est réutilisé. Si le notebook est exécuté dans un nouveau dossier, le texte est téléchargé automatiquement.

In [ ]:
url = "https://www.gutenberg.org/cache/epub/14155/pg14155.txt"
fichier = Path("corpus/madame_bovary_gutenberg_14155.txt")

if not fichier.exists():
    fichier.parent.mkdir(parents=True, exist_ok=True)
    requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(requete) as reponse:
        contenu = reponse.read().decode("utf-8-sig")
    fichier.write_text(contenu, encoding="utf-8")

texte_brut = fichier.read_text(encoding="utf-8")
marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
texte = texte_brut.split(marque_debut, 1)[1].split(marque_fin, 1)[0]

motif_francais = r"[^\W\d_]+(?:['’-][^\W\d_]+)*"
tokens = re.findall(motif_francais, texte.lower())

print(f"{len(tokens):,} tokens")
print(tokens[:30])

## 2. Restreindre le vocabulaire

Avec un vocabulaire de taille $|V|$, une matrice mot-contexte contient $|V|^2$ cellules. Nous retenons les 2 000 tokens les plus fréquents, soit quatre millions de cellules. Une matrice `int32` de cette taille occupe environ 16 Mo ; sa version PPMI en `float64`, environ 32 Mo.

In [ ]:
frequences = Counter(tokens)
vocabulaire = [mot for mot, compte in frequences.most_common(2000)]
ensemble_vocabulaire = set(vocabulaire)

mot_vers_id = {mot: i for i, mot in enumerate(vocabulaire)}
id_vers_mot = {i: mot for mot, i in mot_vers_id.items()}


print(len(vocabulaire))
print(frequences.most_common(20))

`mot_vers_id` associe chaque token à son numéro de ligne ou de colonne. `id_vers_mot` permet de retrouver le token à partir de ce numéro.

### Exercice 1

1. Affichez la fréquence et l'identifiant de `emma`, `charles`, `maison`, `amour` et `cheval`. Vérifiez que ces cinq tokens appartiennent au vocabulaire restreint.

2. Quel est le token le moins fréquent parmi les 2 000 tokens retenus ? Combien de fois apparaît-il ?

## 3. Compter les cooccurrences

Pour chaque position `i`, il faut :

1. lire le token cible `tokens[i]` ;
2. déterminer les bornes de la fenêtre sans sortir de la liste ;
3. parcourir les positions de cette fenêtre ;
4. ignorer la position de la cible elle-même ;
5. incrémenter le compteur si la cible et le contexte appartiennent au vocabulaire.

Voici le calcul complet sur un mini-corpus.

In [ ]:
mini_tokens = "le chat dort le chien dort le chat mange".split()
mini_vocabulaire = set(mini_tokens)
k = 1
mini_comptes = Counter()

for i, cible in enumerate(mini_tokens):
    debut = max(0, i - k)
    fin = min(len(mini_tokens), i + k + 1)

    for j in range(debut, fin):
        if i != j:
            contexte = mini_tokens[j]
            if cible in mini_vocabulaire and contexte in mini_vocabulaire:
                mini_comptes[(cible, contexte)] += 1

print(mini_comptes)

### Exercice 2

1. Écrivez une fonction `cooccurrences(tokens, k, vocabulaire)` qui généralise ce calcul. Elle doit renvoyer un `Counter` dont les clés sont des tuples `(cible, contexte)`.

2. Testez votre fonction sur le mini-corpus avec $k=1$. Vérifiez notamment les comptes de `(chat, le)`, `(chat, dort)` et `(chien, dort)`.

3. Appliquez la fonction au roman avec le vocabulaire restreint et $k=2$. Affichez le nombre de paires distinctes et les vingt paires les plus fréquentes.

## 4. Convertir les comptes en matrice NumPy

`np.zeros((n, n), dtype=np.int32)` crée une matrice carrée initialisée à zéro. Chaque paire comptée doit ensuite être placée à la ligne de la cible et à la colonne du contexte.

Documentation : [`np.zeros`](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html).

### Exercice 3

1. Écrivez une fonction `vers_matrice(comptes, mot_vers_id)` qui crée et renvoie la matrice. Parcourez `comptes.items()` pour récupérer chaque paire et sa fréquence.

2. Convertissez les comptes du roman en matrice `matrice_brute`. Affichez sa forme, son type, sa somme totale et le nombre de cellules non nulles avec `np.count_nonzero()`.

3. Retrouvez dans la matrice le compte de la paire `(emma, charles)` et comparez-le à la valeur du `Counter`.

## 5. Calculer la PPMI

Pour une cellule observée $M_{ij}$, on peut écrire :

$$PMI(i,j) = \log_2\left(\frac{M_{ij} \times N}{M_{i*} \times M_{*j}}\right),$$

où $N$ est la somme de la matrice, $M_{i*}$ la somme de la ligne et $M_{*j}$ la somme de la colonne. La PPMI remplace ensuite toutes les valeurs négatives par zéro.

La fonction suivante applique la formule à toute la matrice. `np.outer()` construit les produits des marges ; `np.log2()` calcule le logarithme ; `np.maximum()` réalise l'écrêtage à zéro.

Documentation : [`np.outer`](https://numpy.org/doc/stable/reference/generated/numpy.outer.html), [`np.log2`](https://numpy.org/doc/stable/reference/generated/numpy.log2.html), [`np.maximum`](https://numpy.org/doc/stable/reference/generated/numpy.maximum.html).

In [ ]:
def calculer_ppmi(matrice):
    matrice = matrice.astype(float)
    total = matrice.sum()
    sommes_lignes = matrice.sum(axis=1)
    sommes_colonnes = matrice.sum(axis=0)
    effectifs_attendus = np.outer(sommes_lignes, sommes_colonnes) / total

    ppmi = np.zeros_like(matrice)
    masque = matrice > 0
    ppmi[masque] = np.log2(matrice[masque] / effectifs_attendus[masque])
    ppmi = np.maximum(ppmi, 0)
    return ppmi

Vérification sur la matrice du CM:

In [ ]:
matrice_exemple = np.array([[4, 0], [1, 3]])
print(np.round(calculer_ppmi(matrice_exemple), 3))

### Exercice 4

1. Calculez `matrice_ppmi` à partir de `matrice_brute`. Affichez sa forme, son type, sa valeur maximale et le nombre de cellules non nulles.

2. Pour `emma`, affichez les dix contextes ayant les comptes bruts les plus élevés, puis les dix contextes ayant les PPMI les plus élevées. Utilisez `np.argsort(ligne)[::-1][:10]` pour obtenir les indices des dix plus grandes valeurs.

## 6. Comparer les plus proches voisins

La fonction suivante calcule en une fois la similarité cosinus entre la ligne d'un token et toutes les lignes de la matrice. Elle utilise le produit matrice-vecteur, `np.linalg.norm()` et `np.argsort()`.

Documentation : [`np.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html), [`np.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html).

In [ ]:
def plus_proches_voisins(matrice, mot, mot_vers_id, id_vers_mot, n=10):
    identifiant = mot_vers_id[mot]
    matrice_float = matrice.astype(float)
    vecteur = matrice_float[identifiant]

    norme_cible = np.linalg.norm(vecteur)
    normes = np.linalg.norm(matrice_float, axis=1)
    denominateurs = normes * norme_cible

    produits_scalaires = matrice_float @ vecteur
    scores = np.divide(
        produits_scalaires,
        denominateurs,
        out=np.zeros_like(normes, dtype=float),
        where=denominateurs != 0,
    )

    scores[identifiant] = -np.inf
    meilleurs_ids = np.argsort(scores)[::-1][:n]
    return [(id_vers_mot[i], float(scores[i])) for i in meilleurs_ids]

### Exercice 5

1. Comparez les dix voisins de `emma`, `charles`, `maison`, `amour` et `cheval` dans la matrice brute et dans la matrice PPMI. Pour quels tokens la pondération modifie-t-elle le plus les résultats ?

2. Recalculez successivement les comptes, la matrice et la PPMI pour $k$ égal à 1, 2, 5 et 10. Pour chaque valeur, affichez les dix voisins PPMI de `maison`.

3. Classez les voisins observés en deux catégories : relations plutôt syntaxiques ou associations plutôt thématiques. La distinction devient-elle plus nette lorsque la fenêtre s'élargit ?